### Saving and loading models, with application to the EuroSat dataset

In [4]:
#importing libraries

import os
import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [5]:
#loading data

def load_eurosat_data():
    data_dir = "data/"
    x_train = np.load(os.path.join(data_dir, "x_test.npy"))
    y_train = np.load(os.path.join(data_dir, "y_test.npy"))

    return (x_train, y_train)

(x_train, y_train)   = load_eurosat_data()
x_train = x_train/255.

In [6]:
print(x_train.shape)

(1000, 64, 64, 3)


In [8]:
#model building

def get_new_model(input_shape):

    model = Sequential([
        Input(shape=input_shape),
        Conv2D(filters=16, kernel_size=(3,3), activation="relu", padding="same"),
        Conv2D(filters=8, kernel_size=(3,3), activation="relu", padding="same"),
        MaxPooling2D(pool_size=(8,8)),
        Flatten(),
        Dense(units=32, activation="relu"),
        Dense(units=10, activation="softmax")

    ])

    model.compile(optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy'])

    return model

In [9]:
model = get_new_model(x_train[0].shape)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 8)      │         1,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 8, 8, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        16,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,354 (71.70 KB)

 Trainable params: 18,354 (71.70 KB)

 Non-trainable params: 0 (0.00 B)

Let's create 3 callbacks:

* checkpoint_every_epoch: checkpoint that saves the model weights every epoch during training
* checkpoint_best_only: checkpoint that saves only the weights with the highest validation accuracy. Use the validation split to get the validation data.
* early_stopping: early stopping object that ends training if the validation accuracy has not improved in 3 epochs

In [13]:
#callback functions

def get_checkpoint_every_epoch():

    epoch_checkpoint = ModelCheckpoint('checkpoint_every_epoch/{epoch:00d}.weights.h5', save_freq="epoch", save_weights_only=True)
    return epoch_checkpoint

def get_checkpoint_best_only():

    best_only_checkpoint = ModelCheckpoint('checkpoint_best_only/.weights.h5', save_weights_only=True, save_best_only=True, monitor="val_accuracy")    
    return best_only_checkpoint


In [15]:
def get_early_stopping():

    early_stopping = EarlyStopping(monitor="val_accuracy", patience=3, mode="max")
    return early_stopping


In [16]:
checkpoint_every_epoch = get_checkpoint_every_epoch()
checkpoint_best_only = get_checkpoint_best_only()
early_stopping = get_early_stopping()